# Prepare data

## Signup date vs first order date

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

DATA_DIR_CANDIDATES = [
    Path("datathon"),
    Path("../datathon"),
    Path.cwd() / "datathon",
    Path.cwd().parent / "datathon",
]
DATA_DIR = next(path for path in DATA_DIR_CANDIDATES if (path / "orders.csv").exists())
DATA_DIR

In [ ]:
customers = pd.read_csv(
    DATA_DIR / "customers.csv",
    usecols=["customer_id", "signup_date", "acquisition_channel"],
    parse_dates=["signup_date"],
)

orders = pd.read_csv(
    DATA_DIR / "orders.csv",
    usecols=["order_id", "customer_id", "order_date"],
    parse_dates=["order_date"],
)

first_order_by_customer = (
    orders.groupby("customer_id", as_index=False)
    .agg(
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max"),
        order_count=("order_id", "count"),
    )
)

signup_first_order = customers.merge(first_order_by_customer, on="customer_id", how="inner")
signup_first_order["signup_minus_first_order_days"] = (
    signup_first_order["signup_date"] - signup_first_order["first_order_date"]
).dt.days
signup_first_order["signup_after_first_order"] = signup_first_order["signup_minus_first_order_days"] > 0
signup_first_order["signup_after_last_order"] = signup_first_order["signup_date"] > signup_first_order["last_order_date"]

summary = pd.DataFrame(
    {
        "customers_with_orders": [len(signup_first_order)],
        "signup_after_first_order_customers": [int(signup_first_order["signup_after_first_order"].sum())],
        "signup_after_first_order_pct": [signup_first_order["signup_after_first_order"].mean() * 100],
        "signup_after_last_order_customers": [int(signup_first_order["signup_after_last_order"].sum())],
        "signup_after_last_order_pct": [signup_first_order["signup_after_last_order"].mean() * 100],
        "median_signup_minus_first_order_days": [signup_first_order["signup_minus_first_order_days"].median()],
    }
)

display(summary.round(2))
display(
    signup_first_order["signup_minus_first_order_days"]
    .describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame("days")
    .round(2)
)

In [ ]:
plot_data = signup_first_order.dropna(subset=["signup_minus_first_order_days"]).copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2.2, 1]})

axes[0].hist(plot_data["signup_minus_first_order_days"], bins=80, color="#4E79A7", edgecolor="white")
axes[0].axvline(0, color="#E15759", linewidth=2, label="signup = first order")
axes[0].set_title("Signup date minus first order date")
axes[0].set_xlabel("days: signup_date - first_order_date")
axes[0].set_ylabel("customers")
axes[0].legend(frameon=False)

axes[1].boxplot(
    plot_data["signup_minus_first_order_days"],
    vert=False,
    patch_artist=True,
    boxprops={"facecolor": "#F28E2B", "edgecolor": "#444444"},
    medianprops={"color": "#111111", "linewidth": 2},
)
axes[1].axvline(0, color="#E15759", linewidth=2)
axes[1].set_title("Distribution spread")
axes[1].set_xlabel("days")
axes[1].set_yticks([])

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
bins = [-10_000, -365, -90, -30, -7, -1, 0, 1, 7, 30, 90, 180, 365, 730, 10_000]
labels = [
    "signup >= 1y before first order",
    "signup 91-365d before",
    "signup 31-90d before",
    "signup 8-30d before",
    "signup 1-7d before",
    "same day",
    "signup 1d after",
    "signup 2-7d after",
    "signup 8-30d after",
    "signup 31-90d after",
    "signup 91-180d after",
    "signup 181-365d after",
    "signup 366-730d after",
    "signup >730d after",
]

diff_bucket_summary = (
    plot_data.assign(
        signup_first_order_gap_bucket=pd.cut(
            plot_data["signup_minus_first_order_days"],
            bins=bins,
            labels=labels,
            include_lowest=True,
        )
    )
    .groupby("signup_first_order_gap_bucket", observed=False)
    .agg(customers=("customer_id", "count"), avg_orders_per_customer=("order_count", "mean"))
    .assign(customer_pct=lambda df: df["customers"] / df["customers"].sum() * 100)
    .reset_index()
)

display(diff_bucket_summary.round({"avg_orders_per_customer": 2, "customer_pct": 2}))

## Định nghĩa metric Revenue, COGS và Profit

Quyết định sử dụng trong EDA này:

- Dùng `sales.Revenue` làm baseline **doanh thu gộp theo ngày**.
- Dùng `sales.COGS` làm baseline **giá vốn theo ngày**.
- Định nghĩa `gross_profit = Revenue - COGS` và `gross_margin_pct = gross_profit / Revenue` cho phân tích doanh thu/lợi nhuận theo ngày.
- Không dùng `payment_value` để thay thế trực tiếp cho `sales.Revenue`.
- Không xem `net_revenue = quantity * unit_price - discount_amount` là cùng một metric với `sales.Revenue`.
- Các tổng transaction sau khi lọc status, ví dụ delivered-only hoặc non-cancelled orders, được xem là một góc nhìn phân tích riêng chứ không phải định nghĩa của `sales.csv`.

Bằng chứng từ bước kiểm tra data quality:

- `sales.Revenue` khớp chính xác với all-order `gross_revenue = quantity * unit_price`, với `0` ngày bị lệch.
- `sales.COGS` khớp với all-order `calculated_cogs = quantity * products.cogs` trong sai số làm tròn, cũng với `0` ngày bị lệch.
- `sales.Revenue` khác `net_revenue` trên `1,707` ngày vì `sales.csv` lưu doanh thu gộp trước discount.
- `sales.Revenue` khác `payment_value` trên `1,707` ngày; `payment_value` đi theo giá trị net/discounted order value.
- `payment_value` khớp chính xác với `net_revenue`, với `0` order bị lệch và max absolute difference chỉ ở mức nhiễu floating-point.
- Các subset transaction theo status khác với `sales.csv`; điều này là expected vì `sales.csv` có vẻ được tạo từ **tất cả đơn hàng**.


## Aggregate `order_items` về grain `order_id + product_id`

Quyết định: không drop các dòng trùng `order_id + product_id`, vì các dòng này không phải exact duplicate. Cách xử lý đúng hơn là aggregate về grain cần dùng cho phân tích product-in-order.

Logic aggregate:

- `quantity`: cộng tổng.
- `gross_revenue`: cộng tổng `quantity * unit_price`.
- `discount_amount`: cộng tổng.
- `net_revenue`: `gross_revenue - discount_amount`.
- `weighted_unit_price`: `gross_revenue / quantity`, thay vì lấy trung bình đơn giản của `unit_price`.
- `line_count`: số dòng gốc được gom lại; nếu `line_count > 1` thì đó là nhóm từng bị duplicate theo candidate key.
- `promo_id`, `promo_id_2`: giữ giá trị non-null đầu tiên và số lượng giá trị unique để biết nhóm có nhiều promotion khác nhau hay không.

In [8]:
order_items_raw = pd.read_csv(DATA_DIR / "order_items.csv", low_memory=False)

order_items_prepared = order_items_raw.assign(
    gross_revenue=lambda df: df["quantity"] * df["unit_price"],
    net_revenue=lambda df: df["quantity"] * df["unit_price"] - df["discount_amount"],
)

order_product_items = (
    order_items_prepared.groupby(["order_id", "product_id"], as_index=False)
    .agg(
        line_count=("product_id", "size"),
        quantity=("quantity", "sum"),
        gross_revenue=("gross_revenue", "sum"),
        discount_amount=("discount_amount", "sum"),
        net_revenue=("net_revenue", "sum"),
        unit_price_min=("unit_price", "min"),
        unit_price_max=("unit_price", "max"),
        promo_id=("promo_id", "first"),
        promo_id_unique_count=("promo_id", "nunique"),
        promo_id_2=("promo_id_2", "first"),
        promo_id_2_unique_count=("promo_id_2", "nunique"),
    )
    .assign(
        weighted_unit_price=lambda df: df["gross_revenue"] / df["quantity"],
        has_multiple_lines=lambda df: df["line_count"] > 1,
        has_unit_price_variation=lambda df: df["unit_price_min"] != df["unit_price_max"],
        has_multiple_promos=lambda df: (df["promo_id_unique_count"] > 1) | (df["promo_id_2_unique_count"] > 1),
    )
)

order_product_items.head()

,order_id,product_id,line_count,quantity,gross_revenue,discount_amount,net_revenue,unit_price_min,unit_price_max,promo_id,promo_id_unique_count,promo_id_2,promo_id_2_unique_count,weighted_unit_price,has_multiple_lines,has_unit_price_variation,has_multiple_promos
0,1,2400,1,7,7967.54,0.0,7967.54,1138.22,1138.22,None,0,None,0,1138.22,False,False,False
1,2,609,1,7,71163.75,0.0,71163.75,10166.25,10166.25,None,0,None,0,10166.25,False,False,False
2,3,396,1,3,33660.99,0.0,33660.99,11220.33,11220.33,None,0,None,0,11220.33,False,False,False
3,4,635,1,5,53196.25,0.0,53196.25,10639.25,10639.25,None,0,None,0,10639.25,False,False,False
4,6,1935,1,1,1597.84,0.0,1597.84,1597.84,1597.84,None,0,None,0,1597.84,False,False,False


In [ ]:
aggregation_validation = pd.DataFrame(
    {
        "metric": ["rows", "quantity", "gross_revenue", "discount_amount", "net_revenue"],
        "before_aggregation": [
            len(order_items_prepared),
            order_items_prepared["quantity"].sum(),
            order_items_prepared["gross_revenue"].sum(),
            order_items_prepared["discount_amount"].sum(),
            order_items_prepared["net_revenue"].sum(),
        ],
        "after_aggregation": [
            len(order_product_items),
            order_product_items["quantity"].sum(),
            order_product_items["gross_revenue"].sum(),
            order_product_items["discount_amount"].sum(),
            order_product_items["net_revenue"].sum(),
        ],
    }
)
aggregation_validation["difference"] = (
    aggregation_validation["after_aggregation"] - aggregation_validation["before_aggregation"]
)

display(aggregation_validation)
display(
    order_product_items.query("has_multiple_lines")
    .sort_values(["order_id", "product_id"])
    .head(20)
)